In [121]:
# 판다스 넘파이 가져오기
import pandas as pd
import numpy as np

In [122]:
# 파일 불러오기
order = pd.read_csv('./data/orders.csv')
oi = pd.read_csv('./data/order_items.csv')
prod = pd.read_csv('./data/products.csv')

In [123]:
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    str    
 2   category      500 non-null    str    
 3   price         500 non-null    str    
 4   cost          500 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.5 KB


In [124]:
# orders 파일 정제
print(f'주문일시 결측 주문 건수 : {order["order_datetime"].isna().sum()}')
orders=order.dropna(subset='order_datetime')
print(f'주문일시 결측 제거 후 주문 건수 : {orders.shape[0]}')
orders=orders.drop_duplicates(subset='order_id')
print(f'중복 제거 후 주문건 수 : {orders.shape[0]}')
orders['order_datetime']=pd.to_datetime(orders['order_datetime'],errors="coerce")
orders.info()

주문일시 결측 주문 건수 : 1933
주문일시 결측 제거 후 주문 건수 : 198067
중복 제거 후 주문건 수 : 198067
<class 'pandas.DataFrame'>
Index: 198067 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        198067 non-null  int64         
 1   customer_id     198067 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         198067 non-null  str           
 4   status          198067 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 11.2 MB


In [125]:
#product함수 관련 결측치 및 제거
prod=prod.drop_duplicates()
prod['price']=pd.to_numeric(prod['price'].astype(str)
                            .str.replace(',','').str.replace('원',''))
prod= prod[prod['price']>0]
prod= prod[prod['cost']>0]
prod.info()

<class 'pandas.DataFrame'>
Index: 489 entries, 1 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    489 non-null    int64  
 1   product_name  489 non-null    str    
 2   category      489 non-null    str    
 3   price         489 non-null    float64
 4   cost          489 non-null    float64
dtypes: float64(2), int64(1), str(2)
memory usage: 35.2 KB


In [126]:
# order_item 파일 정제 (1, 주문 아이템 아이디 중복제거)
orderitem=oi.drop_duplicates(subset='order_item_id')
orderitem.info()

<class 'pandas.DataFrame'>
Index: 499880 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     484774 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 26.7 MB


In [127]:
# order_item 파일 정제 (2. 유닛 프라이스 채우기)
orderitem= orderitem.merge(prod[['product_id','price']],how='left',on='product_id')
orderitem['unit_price']= orderitem['unit_price'].fillna(orderitem["price"])

orderitem=orderitem.drop_duplicates()
orderitem=orderitem.drop(columns='price')
orderitem.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     499641 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [ ]:
# orders & order_items 결합
mix_ooi = orderitem.merge(orders,how='left',on='order_id',validate="m:1")
mix_ooi.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   499880 non-null  int64         
 1   order_id        499880 non-null  int64         
 2   product_id      499880 non-null  int64         
 3   quantity        499880 non-null  int64         
 4   unit_price      499641 non-null  float64       
 5   discount        499880 non-null  float64       
 6   customer_id     495100 non-null  float64       
 7   order_datetime  495100 non-null  datetime64[us]
 8   channel         495100 non-null  str           
 9   status          495100 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(4), str(2)
memory usage: 43.5 MB


In [129]:
#주문정보가 존재하지 않는, 주문 품목 갯수 발견, 정확한 주문과 주문일시를 확인이 불가능하므로,
# 제거 예정
print(f'주문에 들어가있지 않은 주문 품목 수 : {mix_ooi["order_datetime"].isna().sum()}')
mix_ooi = mix_ooi.dropna(subset='order_datetime')
print(f'결측치 제거 후 총 주문품목수 : {mix_ooi.shape[0]}')

주문에 들어가있지 않은 주문 품목 수 : 4780
결측치 제거 후 총 주문품목수 : 495100


In [130]:
mix_ooi=mix_ooi.dropna(subset='unit_price')
mix_ooi = mix_ooi[~(mix_ooi['status'].isin(['canceled','returned']))]
# 2024년 7월 이후데이터는 없는데이터 - 제거
mix_ooi=mix_ooi[mix_ooi['order_datetime']<'2024-07-01']

mix_ooi['amount'] = (
    mix_ooi['quantity']*
    mix_ooi['unit_price']
    *(1-mix_ooi['discount'])
)

mix_ooi.info()


<class 'pandas.DataFrame'>
Index: 420168 entries, 0 to 499879
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   420168 non-null  int64         
 1   order_id        420168 non-null  int64         
 2   product_id      420168 non-null  int64         
 3   quantity        420168 non-null  int64         
 4   unit_price      420168 non-null  float64       
 5   discount        420168 non-null  float64       
 6   customer_id     420168 non-null  float64       
 7   order_datetime  420168 non-null  datetime64[us]
 8   channel         420168 non-null  str           
 9   status          420168 non-null  str           
 10  amount          420168 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(4), str(2)
memory usage: 43.0 MB


In [151]:
valid_orders = (
    mix_ooi.groupby(
        [
            'customer_id',
            'order_id',
            'order_datetime',
            'channel'
        ],
        as_index=False
    )
    .agg(
        amount=('amount', 'sum')
    )
    .sort_values(
        [
            'customer_id',
            'order_datetime',
            'order_id'
        ]
    )
)

In [152]:
valid_orders['order_month']= (valid_orders['order_datetime'].dt.to_period('M'))

In [153]:
valid_orders['cohort_month']= (valid_orders.groupby('customer_id')['order_month'].transform('min'))

valid_orders

,customer_id,order_id,order_datetime,channel,amount,order_month,cohort_month
1,1000.0,22744,2024-01-30 08:13:14,web,151770.0,2024-01,2024-01
3,1000.0,131547,2024-02-01 10:46:11,web,193086.0,2024-02,2024-01
4,1000.0,172530,2024-04-09 01:44:40,web,828838.0,2024-04,2024-01
5,1000.0,185067,2024-04-30 04:32:04,web,790549.0,2024-04,2024-01
6,1000.0,193240,2024-05-06 17:39:14,web,130677.0,2024-05,2024-01
...,...,...,...,...,...,...,...
154702,989712.0,151530,2024-05-20 14:24:49,app,6534.0,2024-05,2024-05
154703,989772.0,163119,2024-01-10 04:09:52,app,560837.0,2024-01,2024-01
154704,989922.0,146788,2024-06-29 03:27:02,web,319564.0,2024-06,2024-06
154705,989977.0,31062,2024-01-14 00:14:02,store,25740.0,2024-01,2024-01


In [154]:
valid_orders['period'] = (
    valid_orders['order_month'].dt.month
    - valid_orders['cohort_month'].dt.month
)

valid_orders

,customer_id,order_id,order_datetime,channel,amount,order_month,cohort_month,period
1,1000.0,22744,2024-01-30 08:13:14,web,151770.0,2024-01,2024-01,0
3,1000.0,131547,2024-02-01 10:46:11,web,193086.0,2024-02,2024-01,1
4,1000.0,172530,2024-04-09 01:44:40,web,828838.0,2024-04,2024-01,3
5,1000.0,185067,2024-04-30 04:32:04,web,790549.0,2024-04,2024-01,3
6,1000.0,193240,2024-05-06 17:39:14,web,130677.0,2024-05,2024-01,4
...,...,...,...,...,...,...,...,...
154702,989712.0,151530,2024-05-20 14:24:49,app,6534.0,2024-05,2024-05,0
154703,989772.0,163119,2024-01-10 04:09:52,app,560837.0,2024-01,2024-01,0
154704,989922.0,146788,2024-06-29 03:27:02,web,319564.0,2024-06,2024-06,0
154705,989977.0,31062,2024-01-14 00:14:02,store,25740.0,2024-01,2024-01,0


In [155]:
cohort_count = valid_orders.pivot_table(
    index='cohort_month',
    columns='period',
    values='customer_id',
    aggfunc='nunique'
    )
cohort_count

period,0,1,2,3,4,5
cohort_month,,,,,,
2024-01,4038.0,3308.0,3513.0,3553.0,3611.0,3699.0
2024-02,847.0,609.0,636.0,646.0,665.0,NaN
2024-03,322.0,152.0,151.0,169.0,NaN,NaN
2024-04,182.0,46.0,49.0,NaN,NaN,NaN
2024-05,160.0,2.0,NaN,NaN,NaN,NaN
2024-06,171.0,NaN,NaN,NaN,NaN,NaN


In [156]:
cohort_amount = valid_orders.pivot_table(
    index='cohort_month',
    columns='period',
    values='amount',
    aggfunc='sum'
    )

cohort = cohort_count.merge(cohort_amount,how='left',on=['cohort_month'],suffixes=['_인원수','_매출액'])
cohort

period,0_인원수,1_인원수,2_인원수,3_인원수,4_인원수,5_인원수,0_매출액,1_매출액,2_매출액,3_매출액,4_매출액,5_매출액
cohort_month,,,,,,,,,,,,
2024-01,4038.0,3308.0,3513.0,3553.0,3611.0,3699.0,5.702378e+09,5.805838e+09,7.160412e+09,7.731182e+09,8.780793e+09,9.625959e+09
2024-02,847.0,609.0,636.0,646.0,665.0,NaN,4.607426e+08,4.306933e+08,4.745276e+08,5.405858e+08,5.734826e+08,NaN
2024-03,322.0,152.0,151.0,169.0,NaN,NaN,1.564746e+08,1.122666e+08,1.078254e+08,1.300540e+08,NaN,NaN
2024-04,182.0,46.0,49.0,NaN,NaN,NaN,6.195484e+07,3.644893e+07,2.855688e+07,NaN,NaN,NaN
2024-05,160.0,2.0,NaN,NaN,NaN,NaN,5.029033e+07,1.879766e+06,NaN,NaN,NaN,NaN
2024-06,171.0,NaN,NaN,NaN,NaN,NaN,5.503935e+07,NaN,NaN,NaN,NaN,NaN


In [157]:
valid_orders['first_date'] = (
    valid_orders
    .groupby('customer_id')['order_datetime']
    .transform('min')
)
valid_orders['days'] = (
    valid_orders['order_datetime']
    - valid_orders['first_date']
).dt.days

valid_orders

,customer_id,order_id,order_datetime,channel,amount,order_month,cohort_month,period,first_date,days
1,1000.0,22744,2024-01-30 08:13:14,web,151770.0,2024-01,2024-01,0,2024-01-30 08:13:14,0
3,1000.0,131547,2024-02-01 10:46:11,web,193086.0,2024-02,2024-01,1,2024-01-30 08:13:14,2
4,1000.0,172530,2024-04-09 01:44:40,web,828838.0,2024-04,2024-01,3,2024-01-30 08:13:14,69
5,1000.0,185067,2024-04-30 04:32:04,web,790549.0,2024-04,2024-01,3,2024-01-30 08:13:14,90
6,1000.0,193240,2024-05-06 17:39:14,web,130677.0,2024-05,2024-01,4,2024-01-30 08:13:14,97
...,...,...,...,...,...,...,...,...,...,...
154702,989712.0,151530,2024-05-20 14:24:49,app,6534.0,2024-05,2024-05,0,2024-05-20 14:24:49,0
154703,989772.0,163119,2024-01-10 04:09:52,app,560837.0,2024-01,2024-01,0,2024-01-10 04:09:52,0
154704,989922.0,146788,2024-06-29 03:27:02,web,319564.0,2024-06,2024-06,0,2024-06-29 03:27:02,0
154705,989977.0,31062,2024-01-14 00:14:02,store,25740.0,2024-01,2024-01,0,2024-01-14 00:14:02,0


In [158]:
#30일만 남기기

window30 = valid_orders[
    valid_orders['days']<=30
].copy()

cohort_A = window30[
    window30['cohort_month'].isin([
        pd.Period('2024-01', 'M'),
        pd.Period('2024-02', 'M')
    ])
]

cohort_B = window30[
    window30['cohort_month'].isin([
        pd.Period('2024-04', 'M'),
        pd.Period('2024-05', 'M')
    ])
]

cohort_A.value_counts('days')

days
0     5439
26     698
27     677
25     663
19     652
18     647
5      644
13     635
30     631
10     631
20     631
4      630
12     630
21     622
11     621
6      617
28     613
29     610
17     594
16     593
24     593
1      572
22     571
14     570
9      568
23     565
2      558
15     552
8      551
7      548
3      544
Name: count, dtype: int64

In [159]:
def cohort_metrics(df):
    result = (
        df.groupby('cohort_month')
        .agg(
            고객수=('customer_id', 'nunique'),
            주문수=('order_id', 'nunique'),
            누적순매출=('amount', 'sum')
        )
    )

    order_count_by_customer = (
        df.groupby(['cohort_month', 'customer_id'])['order_id']
        .nunique()
        .reset_index(name='고객별주문수')
    )

    repurchase_rate = (
        order_count_by_customer
        .assign(재구매여부=lambda x: x['고객별주문수'] >= 2)
        .groupby('cohort_month')['재구매여부']
        .mean()
    )

    result['재구매율'] = repurchase_rate
    result['AOV'] = (
        result['누적순매출']
        / result['주문수']
    )

    result['고객당30일누적매출'] = (
        result['누적순매출']
        / result['고객수']
    )

    return result.reset_index()

In [160]:
cohort_metrics(cohort_A)

,cohort_month,고객수,주문수,누적순매출,재구매율,AOV,고객당30일누적매출
0,2024-01,4038,21621,6.702525e+09,0.815998,310000.682392,1.659862e+06
1,2024-02,847,2049,6.233517e+08,0.683589,304222.388482,7.359524e+05


In [161]:
cohort_metrics(cohort_B)

,cohort_month,고객수,주문수,누적순매출,재구매율,AOV,고객당30일누적매출
0,2024-04,182,272,75665362.0,0.247253,278181.477941,415743.747253
1,2024-05,160,164,50998697.0,0.025000,310967.664634,318741.856250


In [166]:
comparison_30 = pd.DataFrame({
    '코호트': ['A', 'B'],

    '고객수': [
        cohort_A['customer_id'].nunique(),
        cohort_B['customer_id'].nunique()
    ],

    '주문수': [
        cohort_A['order_id'].nunique(),
        cohort_B['order_id'].nunique()
    ],

    '누적순매출': [
        cohort_A['amount'].sum(),
        cohort_B['amount'].sum()
    ],

    '재구매율': [
        cohort_A.groupby('customer_id')['order_id']
        .nunique().ge(2).mean(),

        cohort_B.groupby('customer_id')['order_id']
        .nunique().ge(2).mean()
    ],

    'AOV': [
        cohort_A['amount'].sum()
        / cohort_A['order_id'].nunique(),

        cohort_B['amount'].sum()
        / cohort_B['order_id'].nunique()
    ],

    '고객당누적매출': [
        cohort_A['amount'].sum()
        / cohort_A['customer_id'].nunique(),

        cohort_B['amount'].sum()
        / cohort_B['customer_id'].nunique()
    ]
})

comparison_30

,코호트,고객수,주문수,누적순매출,재구매율,AOV,고객당누적매출
0,A,4885,23670,7.325876e+09,0.793040,309500.482805,1.499668e+06
1,B,342,436,1.266641e+08,0.143275,290513.896789,3.703627e+05


In [163]:
#고객 총매출

print(f'A분류 고객 수 :{cohort_A["customer_id"].nunique()},\n A분류 고객 매출합 : {round(cohort_A["amount"].sum(),0)},\n A분류 고객 평균 : {round(cohort_A.groupby("customer_id")["amount"].sum().mean(),0)}')
print(f'B분류 고객 수 : {cohort_B["customer_id"].nunique()},\n B분류 고객 매출합 : {round(cohort_B["amount"].sum(),0)},\n A분류 고객 평균 : {round(cohort_B.groupby("customer_id")["amount"].sum().mean(),0)}')

A분류 고객 수 :4885,
 A분류 고객 매출합 : 7325876428.0,
 A분류 고객 평균 : 1499668.0
B분류 고객 수 : 342,
 B분류 고객 매출합 : 126664059.0,
 A분류 고객 평균 : 370363.0


In [164]:
cohort_A_all = valid_orders[
    valid_orders['cohort_month'].isin([
        pd.Period('2024-01', 'M'),
        pd.Period('2024-02', 'M')
    ])
]

cohort_B_all = valid_orders[
    valid_orders['cohort_month'].isin([
        pd.Period('2024-04', 'M'),
        pd.Period('2024-05', 'M')
    ])
]

In [165]:
comparison_all = pd.DataFrame({
    '코호트': ['A', 'B'],

    '고객수': [
        cohort_A_all['customer_id'].nunique(),
        cohort_B_all['customer_id'].nunique()
    ],

    '주문수': [
        cohort_A_all['order_id'].nunique(),
        cohort_B_all['order_id'].nunique()
    ],

    '누적순매출': [
        cohort_A_all['amount'].sum(),
        cohort_B_all['amount'].sum()
    ],

    '재구매율': [
        cohort_A_all.groupby('customer_id')['order_id']
        .nunique().ge(2).mean(),

        cohort_B_all.groupby('customer_id')['order_id']
        .nunique().ge(2).mean()
    ],

    'AOV': [
        cohort_A_all['amount'].sum()
        / cohort_A_all['order_id'].nunique(),

        cohort_B_all['amount'].sum()
        / cohort_B_all['order_id'].nunique()
    ],

    '고객당누적매출': [
        cohort_A_all['amount'].sum()
        / cohort_A_all['customer_id'].nunique(),

        cohort_B_all['amount'].sum()
        / cohort_B_all['customer_id'].nunique()
    ]
})

comparison_all

,코호트,고객수,주문수,누적순매출,재구매율,AOV,고객당누적매출
0,A,4885,152334,4.728659e+10,0.960901,310413.914149,9.679958e+06
1,B,342,604,1.791307e+08,0.175439,296574.076159,5.237741e+05


## 결론
A코호트는 2024년 1~2월에 처음 구매한 고객, B코호트는 2024년 4~5월에 처음 구매한 고객으로 구분하였다. 두 코호트의 관찰 기간 차이를 제거하기 위해 각 고객의 첫 구매일을 기준으로 30일 차까지의 주문만 비교하였다.

30일 기준으로 A코호트의 고객 수는 4,885명, 누적 순매출은 약 73억 2,588만 원, 고객당 누적 매출은 약 150만 원이었다. B코호트의 고객 수는 342명, 누적 순매출은 약 1억 2,666만 원, 고객당 누적 매출은 약 37만 원이었다.

재구매율은 A코호트가 약 79.3%, B코호트가 약 14.3%로 나타났다. AOV는 A코호트가 약 30만 9,500원, B코호트가 약 29만 500원으로 큰 차이가 없었다.

따라서 A코호트와 B코호트의 성과 차이는 주문 한 건당 금액인 AOV보다는, 첫 구매 후 30일 이내에 다시 구매한 고객의 비율과 고객별 주문 횟수에서 크게 발생한 것으로 판단된다. A코호트는 B코호트보다 고객당 누적 매출과 재구매율이 모두 높으므로, 동일한 30일 기준에서는 A코호트의 고객 품질이 더 높았다고 볼 수 있다.

30일 제한을 적용하지 않은 전체 기간 비교에서는 A코호트의 재구매율이 약 96.1%, 고객당 누적 매출이 약 968만 원으로 더욱 크게 나타났다. 그러나 A코호트는 B코호트보다 먼저 유입되어 더 오랜 기간 관찰되었기 때문에, 전체 기간 결과에는 관찰 기간 차이로 인한 편향이 포함되어 있다.

이에 따라 고객 품질에 대한 최종 판단은 전체 기간 결과가 아니라 동일한 30일 관찰 기간을 적용한 결과를 기준으로 해야 한다. 최종적으로 A코호트의 고객 품질이 B코호트보다 높게 나타났으며, 그 차이의 핵심은 객단가보다 재구매 행동에 있다고 판단된다.

다만 A코호트의 성과가 높았던 구체적인 원인은 현재 데이터만으로 확인할 수 없다. 프로모션, 고객 유입 경로, 상품 구성 등의 영향은 관련 데이터를 추가로 분석해야 판단할 수 있다.